# analysis_b / 01+03 — Posts-Only Pipeline

Combined data-preparation + LDA topic modelling, restricted to **posts only** (`type == 'post'`). Comments are excluded at load time.

**Reads:** `../reddit_scrapy/data/Paranormal_20250101_20251231_timerange.csv`  
**Writes:** `artifacts/posts_only/posts_clean.parquet`, `theta.npy`, `phi.npy`, `lda_vocab.json`, `lda_model.pkl`, `topic_labels.json`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import spacy
import re
import json
import textwrap
import warnings
import joblib
import itertools
from pathlib import Path
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.model_selection import train_test_split
from scipy.sparse import csc_matrix
from scipy.stats import spearmanr, rankdata
from scipy.interpolate import UnivariateSpline
from scipy.stats import linregress
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import r2_score
from tqdm import tqdm

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

DATA_PATH  = Path('../reddit_scrapy/data/Paranormal_20250101_20251231_timerange.csv')
ARTIFACTS  = Path('artifacts/posts_only')
ARTIFACTS.mkdir(parents=True, exist_ok=True)

MIN_TOKENS = 5  # drop posts with fewer lemma tokens than this after spaCy
ARTIFACTS_D2 = Path('artifacts/posts_comments')
ARTIFACTS_D2.mkdir(parents=True, exist_ok=True)
MIN_TOKENS_D2 = 10  # higher floor for combined post+comment docs


## 1. Load & Filter to Posts

In [ ]:
df_raw = pd.read_csv(DATA_PATH)
print(f'Raw shape : {df_raw.shape}')
print(f'type counts:\n{df_raw["type"].value_counts()}')

# Keep posts only — drop all comments
df_raw = df_raw[df_raw['type'] == 'post'].reset_index(drop=True)
print(f'\nAfter post filter: {df_raw.shape}')
print(f'type unique: {df_raw["type"].unique()}')
df_raw.head(3)

## 2. Column Standardization

In [ ]:
COLUMN_MAP = {
    'selftext'       : 'text',
    'body'           : 'text',
    'post_text'      : 'text',
    'score'          : 'score',
    'ups'            : 'score',
    'link_flair_text': 'flair',
    'flair'          : 'flair',
    'post_flair'     : 'flair',
    'created_utc'    : 'created_utc',
    'created'        : 'created_utc',
    'timestamp'      : 'created_utc',
    'id'             : 'post_id',
    'post_id'        : 'post_id',
    'title'          : 'title',
}

df = df_raw.rename(columns={k: v for k, v in COLUMN_MAP.items() if k in df_raw.columns})

required = ['text', 'created_utc']
missing  = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(
        f'Missing required columns after rename: {missing}.\n'
        f'Available columns: {df.columns.tolist()}\n'
        f'Update COLUMN_MAP above to match your CSV headers.'
    )

for col, default in [('score', np.nan), ('flair', 'unknown'), ('title', ''), ('post_id', None)]:
    if col not in df.columns:
        df[col] = default

print('Standardized columns:', df.columns.tolist())
print(f'Total rows: {len(df):,}')

## 3. Metadata Overview

In [ ]:
if df['score'].notna().any():
    cap = df['score'].quantile(0.99)
    fig, ax = plt.subplots(figsize=(6, 4))
    df['score'].clip(upper=cap).plot(kind='hist', bins=50, ax=ax)
    ax.set_title(f'Score distribution (clipped at 99th pct = {cap:.0f})')
    ax.set_xlabel('Score')
    plt.tight_layout()
    plt.show()
    print(df['score'].describe())
else:
    print('No score data available.')

## 4. Text Cleaning

Strip Reddit-specific markup. Original text preserved in `text_raw`.

In [ ]:
def clean_text(text):
    if not isinstance(text, str) or not text.strip():
        return ''
    if text.strip().lower() in ('[deleted]', '[removed]'):
        return ''
    text = re.sub(r'(?m)^>.*$', '', text)            # blockquotes
    text = re.sub(r'https?://\S+|www\.\S+', '', text) # URLs
    text = re.sub(r'\[([^\]]+)\]\([^\)]*\)', r'\1', text)  # markdown links
    text = re.sub(r'\*{1,3}([^*]+)\*{1,3}', r'\1', text)  # bold/italic
    text = re.sub(r'~~([^~]+)~~', r'\1', text)       # strikethrough
    text = re.sub(r'```[\s\S]*?```', '', text)        # fenced code
    text = re.sub(r'`[^`]+`', '', text)               # inline code
    text = re.sub(r'(?m)^#{1,6}\s', '', text)         # ATX headers
    text = re.sub(r'\bu/\w+', '', text)               # user mentions
    text = re.sub(r'\br/\w+', '', text)               # subreddit mentions
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

df['text_raw']   = df['text'].copy()
df['text_clean'] = df['text_raw'].apply(clean_text)

before = len(df)
df = df[df['text_clean'].str.len() > 0].reset_index(drop=True)
print(f'Dropped {before - len(df):,} empty/deleted posts. Remaining: {len(df):,}')

In [ ]:
# Most frequent authors
print(df['author'].value_counts().head(20))

In [ ]:
# Drop AutoModerator posts
before = len(df)
df = df[df['author'] != 'AutoModerator'].reset_index(drop=True)
print(f'Dropped {before - len(df):,} AutoModerator posts. Remaining: {len(df):,}')

## 5. spaCy Lemmatization

Pipeline: tokenize → POS-filter (NOUN, VERB, ADJ, ADV) → remove stopwords and short tokens → lemmatize. Posts below `MIN_TOKENS` after this step are dropped.

In [ ]:
pip install https://github.com/explosion/spacy-models/releases/download/en_core_web_lg-3.8.0/en_core_web_lg-3.8.0-py3-none-any.whl

In [ ]:
nlp = spacy.load('en_core_web_lg')
nlp.select_pipes(enable=['tok2vec', 'tagger', 'attribute_ruler', 'lemmatizer'])

KEEP_POS = {'NOUN', 'VERB', 'ADJ', 'ADV'}

def extract_tokens(doc):
    return [
        token.lemma_.lower()
        for token in doc
        if  token.pos_  in KEEP_POS
        and not token.is_stop
        and not token.is_punct
        and not token.like_num
        and not token.is_space
        and len(token.lemma_) > 2
    ]

In [ ]:
raw_word_count = df['text_clean'].str.split().apply(len)

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(raw_word_count, bins=80, edgecolor='black')
ax.axvline(MIN_TOKENS, color='red', linestyle='--', label=f'MIN_TOKENS = {MIN_TOKENS}')
ax.set_xlabel('Words per document (pre-lemmatization)')
ax.set_ylabel('Number of documents')
ax.set_title('Document length distribution (raw word count)')
ax.legend()
plt.tight_layout()
plt.show()

print(raw_word_count.describe())
print(f'\nDocs below MIN_TOKENS={MIN_TOKENS}: {(raw_word_count < MIN_TOKENS).sum():,} ({(raw_word_count < MIN_TOKENS).mean():.1%})')

In [ ]:
BATCH_SIZE = 64
texts      = df['text_clean'].tolist()

all_tokens = [
    extract_tokens(doc)
    for doc in nlp.pipe(texts, batch_size=BATCH_SIZE)
]

df['tokens']      = all_tokens
df['text_lemma']  = df['tokens'].apply(' '.join)
df['token_count'] = df['tokens'].apply(len)

before = len(df)
df = df[df['token_count'] >= MIN_TOKENS].reset_index(drop=True)
print(f'Dropped {before - len(df):,} posts below {MIN_TOKENS}-token floor. Remaining: {len(df):,}')
print(f'Median token count: {df["token_count"].median():.0f}')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
df['token_count'].clip(upper=500).plot(kind='hist', bins=60, ax=ax)
ax.set_title('Token count per post after lemmatization (clipped at 500)')
ax.set_xlabel('Tokens')
plt.tight_layout()
plt.show()

## 6. Temporal Features

Parses `created_utc`, extracts `year_month`, `year`, `month`, and **`date`** (calendar day) and **`day_idx`** (integer 0 → N−1).

In [ ]:
df['created_utc'] = pd.to_datetime(df['created_utc'], errors='coerce')

n_bad = df['created_utc'].isnull().sum()
if n_bad > 0:
    print(f'Warning: {n_bad} unparseable timestamps — dropping.')
    df = df.dropna(subset=['created_utc']).reset_index(drop=True)

df['year_month'] = df['created_utc'].dt.to_period('M').astype(str)
df['year']       = df['created_utc'].dt.year
df['month']      = df['created_utc'].dt.month
df['date']       = df['created_utc'].dt.normalize()
df['day_idx']    = (df['date'] - df['date'].min()).dt.days

print(f'Date range : {df["date"].min().date()} → {df["date"].max().date()}')
print(f'Span       : {int(df["day_idx"].max()) + 1} days')
print('Months present:', sorted(df['year_month'].unique()))

In [ ]:
daily_counts = df.groupby('date').size().sort_index()

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(range(len(daily_counts)), daily_counts.values, color='steelblue', width=0.8)
ax.set_xticks(range(len(daily_counts)))
ax.set_xticklabels(
    [d.strftime('%b %d') for d in daily_counts.index], rotation=45, ha='right'
)
ax.set_title('Posts per day (posts only, after cleaning + token filter)')
ax.set_xlabel('Date')
ax.set_ylabel('Posts')
plt.tight_layout()
plt.show()

## 7. Write Artifact

Writes `artifacts/posts_only/posts_clean.parquet`.

| Column | Type | Notes |
|---|---|---|
| `post_id` | str | Reddit post ID |
| `post_title` | str | Post title |
| `text_raw` | str | Original scraped text |
| `text_clean` | str | Markup-stripped text |
| `text_lemma` | str | Space-joined lemmas |
| `tokens` | list[str] | Lemma token list |
| `token_count` | int | Post length in tokens |
| `score` | float | Upvote score |
| `flair` | str | Post flair |
| `created_utc` | datetime | UTC post time |
| `year_month` | str | e.g. `'2025-01'` |
| `year` | int | |
| `month` | int | |
| `date` | datetime | Calendar day (midnight) |
| `day_idx` | int | Days since first post (0-based) |

In [ ]:
SAVE_COLS = [
    'post_id', 'type', 'post_title', 'text_raw', 'text_clean', 'text_lemma',
    'tokens', 'token_count', 'score', 'flair',
    'created_utc', 'year_month', 'year', 'month', 'date', 'day_idx',
]
SAVE_COLS = [c for c in SAVE_COLS if c in df.columns]

out_path = ARTIFACTS / 'posts_clean.parquet'
df[SAVE_COLS].to_parquet(out_path, index=False)

print(f'Saved {len(df):,} posts → {out_path}')
print(f'Columns : {SAVE_COLS}')
print(f'File size: {out_path.stat().st_size / 1024:.1f} KB')

---
## Part 1b — Dataset 2: Posts + Aggregated Comments

Each document = post selftext lemmas + all comment lemmas for that `post_id`.

**Purpose:** sLDA-C in Part 4 needs richer documents that incorporate community discussion. Engagement metrics (`log_score`, `log_num_comments`) are stored as continuous response variables.

**Reads:** same `DATA_PATH` CSV as Part 1
**Writes:** `artifacts/posts_comments/posts_with_comments.parquet`

In [ ]:
# ── load ALL rows; keep comments this time ────────────────────────────────────
_df_all = pd.read_csv(DATA_PATH)
df_comments_raw = _df_all[_df_all['type'] == 'comment'].reset_index(drop=True)

# Standardise column names (same COLUMN_MAP as posts)
df_comments_raw = df_comments_raw.rename(
    columns={k: v for k, v in COLUMN_MAP.items() if k in df_comments_raw.columns}
)

# Fill missing columns with defaults
for col, default in [('score', np.nan), ('flair', 'unknown'), ('post_id', None)]:
    if col not in df_comments_raw.columns:
        df_comments_raw[col] = default

# Assign text column (comments use 'body')
if 'text' not in df_comments_raw.columns:
    raise ValueError('Cannot find body/text column in comments — update COLUMN_MAP.')

# Clean comment text
df_comments_raw['text_clean'] = df_comments_raw['text'].apply(clean_text)
df_comments_raw = df_comments_raw[df_comments_raw['text_clean'].str.len() > 0]

# Drop AutoModerator comments
if 'author' in df_comments_raw.columns:
    df_comments_raw = df_comments_raw[df_comments_raw['author'] != 'AutoModerator']

print(f'Comment rows after cleaning: {len(df_comments_raw):,}')
print(f'Unique posts with comments : {df_comments_raw["post_id"].nunique():,}')
df_comments_raw.head(2)

In [ ]:
# ── Lemmatise comments — skip if cached parquet exists ────────────────────────
_cache = ARTIFACTS_D2 / 'posts_with_comments.parquet'

if _cache.exists():
    df2 = pd.read_parquet(_cache)
    print(f'Loaded cached Dataset 2 from {_cache}  ({len(df2):,} rows)')
    _d2_cached = True
else:
    _d2_cached = False
    comment_texts = df_comments_raw['text_clean'].tolist()
    comment_tokens_list = [
        extract_tokens(doc)
        for doc in tqdm(
            nlp.pipe(comment_texts, batch_size=128),
            total=len(comment_texts),
            desc='Lemmatising comments',
        )
    ]
    df_comments_raw['tokens'] = comment_tokens_list
    print(f'Lemmatised {len(comment_tokens_list):,} comment texts')

In [ ]:
if not _d2_cached:
    # ── Aggregate comment tokens per post_id ──────────────────────────────────
    comment_agg = (
        df_comments_raw
        .groupby('post_id')['tokens']
        .apply(lambda series: [tok for tl in series for tok in tl])
        .reset_index()
        .rename(columns={'tokens': 'comment_tokens'})
    )
    print(f'Posts with at least one comment: {len(comment_agg):,}')
    print(f'Median comment tokens per post : '
          f'{comment_agg["comment_tokens"].apply(len).median():.0f}')

In [ ]:
if not _d2_cached:
    # ── Merge comment tokens onto Dataset 1 posts ─────────────────────────────
    _keep = ['post_id', 'tokens', 'text_lemma', 'score', 'flair',
             'created_utc', 'year_month', 'year', 'month', 'date', 'day_idx']
    _keep = [c for c in _keep if c in df.columns]

    df2 = df[_keep].copy()
    df2 = df2.merge(comment_agg, on='post_id', how='left')
    df2['comment_tokens'] = df2['comment_tokens'].apply(
        lambda x: x if isinstance(x, list) else []
    )

    # Combined document = post tokens + comment tokens
    df2['combined_tokens']      = df2['tokens'] + df2['comment_tokens']
    df2['combined_text_lemma']  = df2['combined_tokens'].apply(' '.join)
    df2['combined_token_count'] = df2['combined_tokens'].apply(len)

    # Response variables — use num_comments from post rows in _df_all
    _post_meta = (
        _df_all[_df_all['type'] == 'post']
        .rename(columns={k: v for k, v in COLUMN_MAP.items() if k in _df_all.columns})
        .set_index('post_id')
    )
    _nc_col = next(
        (c for c in ['num_comments', 'num_comment', 'comments'] if c in _post_meta.columns),
        None
    )
    df2['num_comments'] = (
        df2['post_id'].map(_post_meta[_nc_col]) if _nc_col else np.nan
    )
    df2['log_score']        = np.log1p(df2['score'].fillna(0))
    df2['log_num_comments'] = np.log1p(df2['num_comments'].fillna(0))

    # Apply higher token floor
    before2 = len(df2)
    df2 = df2[df2['combined_token_count'] >= MIN_TOKENS_D2].reset_index(drop=True)
    print(f'Dropped {before2 - len(df2):,} posts below {MIN_TOKENS_D2}-token floor.')
    print(f'Dataset 2: {len(df2):,} posts')

In [ ]:
# ── Compare Dataset 1 vs Dataset 2 ────────────────────────────────────────────
print(f'Dataset 1 (posts-only)  : {len(df):,} posts, '
      f'median tokens = {df["token_count"].median():.0f}')
print(f'Dataset 2 (posts+comments): {len(df2):,} posts, '
      f'median combined tokens = {df2["combined_token_count"].median():.0f}')
print()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].hist(df2['combined_token_count'].clip(upper=2000), bins=60, color='steelblue', edgecolor='none')
axes[0].set_title('Combined token count per post (Dataset 2)')
axes[0].set_xlabel('Tokens')
axes[0].set_ylabel('Posts')

axes[1].hist(df2['log_score'], bins=40, color='coral', alpha=0.7, label='log_score', edgecolor='none')
axes[1].hist(df2['log_num_comments'], bins=40, color='mediumseagreen', alpha=0.7,
             label='log_num_comments', edgecolor='none')
axes[1].set_title('Response variable distributions (Dataset 2)')
axes[1].set_xlabel('log1p value')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
if not _d2_cached:
    _save_cols2 = [
        'post_id', 'score', 'num_comments', 'log_score', 'log_num_comments',
        'flair', 'text_lemma', 'combined_text_lemma', 'combined_tokens',
        'combined_token_count', 'created_utc', 'year_month', 'year', 'month',
        'date', 'day_idx',
    ]
    _save_cols2 = [c for c in _save_cols2 if c in df2.columns]
    df2[_save_cols2].to_parquet(ARTIFACTS_D2 / 'posts_with_comments.parquet', index=False)
    print(f'Saved {len(df2):,} rows -> {ARTIFACTS_D2 / "posts_with_comments.parquet"}')
    print(f'Columns: {_save_cols2}')
else:
    print('Dataset 2 already cached — skipped save.')

---
# Part 2 — Latent Topic Discovery (LDA)

Unsupervised discovery of latent thematic structure in the posts-only corpus. No pre-imposed categories — topics emerge from the data.

**Pipeline:** DTM → K selection (perplexity + PMI coherence) → fit final LDA → topic profiling (FREX + top documents) → score correlations

## 8. Document-Term Matrix

In [ ]:
vec = CountVectorizer(min_df=10, max_df=0.90)
dtm = vec.fit_transform(df['text_lemma'])
vocab = vec.get_feature_names_out()

print(f'DTM      : {dtm.shape[0]:,} docs × {dtm.shape[1]:,} terms')
print(f'Sparsity : {1 - dtm.nnz / (dtm.shape[0] * dtm.shape[1]):.2%}')

X_train, X_test = train_test_split(dtm, test_size=0.10, random_state=42)
print(f'Train: {X_train.shape[0]:,}   Test: {X_test.shape[0]:,}')

## 9. K Selection

- **Perplexity** on held-out 10% — lower is better; tends to keep decreasing, so not sufficient alone
- **PMI coherence** — higher means more semantically coherent topics; choose K at the elbow

In [ ]:
dtm_csc = csc_matrix((dtm > 0).astype(np.float32))

def pmi_coherence(topic_idx, phi, dtm_csc, n_top=10):
    top_w = phi[topic_idx].argsort()[::-1][:n_top]
    cols  = [
        np.asarray(dtm_csc.getcol(int(w)).todense()).flatten() > 0
        for w in top_w
    ]
    pmis = []
    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            p_i  = cols[i].mean()
            p_j  = cols[j].mean()
            p_ij = (cols[i] & cols[j]).mean()
            if p_i > 0 and p_j > 0 and p_ij > 0:
                pmis.append(np.log(p_ij / (p_i * p_j)))
    return float(np.mean(pmis)) if pmis else 0.0

In [ ]:
K_RANGE = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
results = []

for K in K_RANGE:
    lda_k = LatentDirichletAllocation(
        n_components=K, learning_method='online', max_iter=20, random_state=42
    )
    lda_k.fit(X_train)
    perp  = lda_k.perplexity(X_test)
    phi_k = lda_k.components_ / lda_k.components_.sum(axis=1, keepdims=True)
    coh   = float(np.mean([pmi_coherence(k, phi_k, dtm_csc) for k in range(K)]))
    results.append({'K': K, 'perplexity': perp, 'coherence': coh})
    print(f'K={K:2d}  perplexity={perp:9.1f}  coherence={coh:.4f}')

results_df = pd.DataFrame(results)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.plot(results_df['K'], results_df['perplexity'], marker='o')
ax1.set_title('Perplexity on held-out 10%')
ax1.set_xlabel('K')
ax1.set_ylabel('Perplexity (lower = better)')

ax2.plot(results_df['K'], results_df['coherence'], marker='o', color='steelblue')
ax2.set_title('Mean PMI Coherence')
ax2.set_xlabel('K')
ax2.set_ylabel('Mean pairwise PMI (higher = better)')

plt.suptitle('K selection — choose K at the coherence elbow')
plt.tight_layout()
plt.show()

## 10. Fit Final Model

Set `K_FINAL` based on the plot above.

In [ ]:
K_FINAL = 4

lda = LatentDirichletAllocation(
    n_components=K_FINAL, learning_method='batch', max_iter=100, random_state=42
)
lda.fit(dtm)

print(f'Final model: K={K_FINAL}, held-out perplexity={lda.perplexity(X_test):.1f}')

In [ ]:
# phi   : (K, V) normalized topic-word matrix
# theta : (N, K) document-topic distribution
phi   = lda.components_ / lda.components_.sum(axis=1, keepdims=True)
theta = lda.transform(dtm)

print(f'phi   shape : {phi.shape}')
print(f'theta shape : {theta.shape}')
print(f'theta row-sum check (should be ~1.0): {theta.sum(axis=1).mean():.4f}')

## 11. Topic Profiling

For each topic: top words + top documents by theta loading. Read the documents — do not rely on words alone.

In [ ]:
N_TOP_WORDS = 20
vocab_list  = list(vocab)

print('Top words per topic')
print('=' * 70)
for k in range(K_FINAL):
    top_idx   = phi[k].argsort()[::-1][:N_TOP_WORDS]
    top_words = ', '.join(vocab_list[i] for i in top_idx)
    print(f'Topic {k:2d}: {top_words}')

In [ ]:
N_TOP = 20

top_word_sets = {}
for k in range(K_FINAL):
    top_idx = phi[k].argsort()[::-1][:N_TOP]
    top_word_sets[k] = set(vocab_list[i] for i in top_idx)

overlap = np.zeros((K_FINAL, K_FINAL), dtype=int)
for i, j in itertools.combinations(range(K_FINAL), 2):
    shared = top_word_sets[i] & top_word_sets[j]
    overlap[i, j] = len(shared)
    overlap[j, i] = len(shared)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    overlap, annot=True, fmt='d', cmap='YlOrRd',
    xticklabels=[f'T{k}' for k in range(K_FINAL)],
    yticklabels=[f'T{k}' for k in range(K_FINAL)],
    ax=ax
)
ax.set_title(f'Shared top-{N_TOP} words between topic pairs')
plt.tight_layout()
plt.show()

print(f'\nShared words (top-{N_TOP}) per pair:')
for i, j in itertools.combinations(range(K_FINAL), 2):
    shared = sorted(top_word_sets[i] & top_word_sets[j])
    print(f'  T{i} ∩ T{j} ({len(shared)}): {", ".join(shared) if shared else "none"}')

### FREX — Frequency × Exclusivity

Top words by raw probability alone are dominated by corpus-wide filler (*think*, *see*, *people*).
FREX re-ranks words by the harmonic mean of two ECDF-normalised scores:

- **Frequency** `phi[k, w]` — probability of the word within the topic  
- **Exclusivity** `phi[k, w] / Σ_k phi[k, w]` — share of the word's total mass in this topic

A word scores high only when it is *both* common in the topic *and* concentrated there.

In [ ]:
def frex(phi, w=0.5, n_top=15):
    K, V = phi.shape
    col_sums = phi.sum(axis=0)
    excl = phi / np.where(col_sums > 0, col_sums, 1)
    freq_ecdf = np.apply_along_axis(
        lambda row: rankdata(row) / V, axis=1, arr=phi
    )
    excl_ecdf = np.apply_along_axis(
        lambda row: rankdata(row) / V, axis=1, arr=excl
    )
    scores = 1.0 / (w / freq_ecdf + (1 - w) / excl_ecdf)
    top_idx = scores.argsort(axis=1)[:, ::-1][:, :n_top]
    return scores, top_idx


N_FREX   = 15
FREX_W   = 0.5

frex_scores, frex_top = frex(phi, w=FREX_W, n_top=N_FREX)

print(f'FREX top-{N_FREX} words per topic  (w={FREX_W})')
print('=' * 70)
for k in range(K_FINAL):
    words = ', '.join(vocab_list[i] for i in frex_top[k])
    print(f'Topic {k}: {words}')

In [ ]:
N_CMP = 10
cols = pd.MultiIndex.from_product(
    [[f'Topic {k}' for k in range(K_FINAL)], ['Raw', 'FREX']]
)
data = {}
for k in range(K_FINAL):
    raw_words  = [vocab_list[i] for i in phi[k].argsort()[::-1][:N_CMP]]
    frex_words = [vocab_list[i] for i in frex_top[k][:N_CMP]]
    data[(f'Topic {k}', 'Raw')]  = raw_words
    data[(f'Topic {k}', 'FREX')] = frex_words

cmp_df = pd.DataFrame(data, index=range(1, N_CMP + 1))
cmp_df.index.name = 'Rank'
cmp_df

### Doc Inspection

In [ ]:
N_TOP_DOCS = 10

for k in range(K_FINAL):
    print('=' * 70)
    print(f'TOPIC {k} — top {N_TOP_DOCS} documents')
    print('=' * 70)
    top_doc_idx = theta[:, k].argsort()[::-1][:N_TOP_DOCS]
    for rank, idx in enumerate(top_doc_idx, start=1):
        row     = df.iloc[idx]
        loading = theta[idx, k]
        snippet = row['text_clean']
        date    = str(row.get('date', row.get('year_month', '')))
        print(f'  [{rank}] theta={loading:.3f} | {date}')
        print(textwrap.fill(snippet, width=76, initial_indent='      ', subsequent_indent='      '))
        print()
    print()

In [ ]:
N_TOP_DOCS = 10
md_path    = ARTIFACTS / 'top_docs.md'

lines = []
for k in range(K_FINAL):
    lines.append(f'# TOPIC {k} — top {N_TOP_DOCS} documents\n\n')
    top_doc_idx = theta[:, k].argsort()[::-1][:N_TOP_DOCS]
    for rank, idx in enumerate(top_doc_idx, start=1):
        row     = df.iloc[idx]
        loading = theta[idx, k]
        snippet = str(row['text_clean'])
        date    = str(row.get('date', row.get('year_month', '')))
        lines.append(f'- [{rank}] theta={loading:.3f} | {date}\n\n')
        wrapped = textwrap.fill(snippet, width=76,
                                initial_indent='  ',
                                subsequent_indent='  ')
        lines.append(f'{wrapped}\n\n')
    lines.append('\n---\n\n')

md_path.write_text(''.join(lines), encoding='utf-8')
print(f'Written → {md_path}')

In [ ]:
N_SAMPLE    = 5
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

W = 82

dominant = theta.argmax(axis=1)

for k in range(K_FINAL):
    print('\n' + '─' * W)
    print(f'  TOPIC {k} — {N_SAMPLE} sampled posts (dominant-member only)')
    print('─' * W)

    members    = np.where(dominant == k)[0]
    sample_idx = rng.choice(members, size=min(N_SAMPLE, len(members)), replace=False)

    for rank, idx in enumerate(sample_idx, start=1):
        row     = df.iloc[idx]
        loading = theta[idx, k]
        date    = str(row.get('date', row.get('year_month', '')))

        print(f'\n  [{rank}]  {date}  |  score={row["score"]}  |  theta={loading:.3f}')
        print('  ' + '─' * (W - 2))

        paragraph = textwrap.fill(str(row['text_clean']), width=W - 4,
                                  initial_indent='  ',
                                  subsequent_indent='  ')
        print(paragraph)
        print()

## 12. Topic Distribution Heatmap

Mean theta per topic across the full corpus — shows which topics dominate overall.

In [ ]:
mean_theta = theta.mean(axis=0)
topic_labels_tmp = [f'Topic {k}' for k in range(K_FINAL)]

fig, ax = plt.subplots(figsize=(10, 3))
ax.bar(range(K_FINAL), mean_theta, color='steelblue')
ax.set_xticks(range(K_FINAL))
ax.set_xticklabels(topic_labels_tmp, rotation=45, ha='right')
ax.set_title('Mean theta loading per topic (corpus-wide, posts only)')
ax.set_ylabel('Mean theta')
plt.tight_layout()
plt.show()

## 13. Score Correlation

Spearman rho between post score and each topic's theta loading. Bonferroni-corrected.

In [ ]:
score_vals  = df['score'].fillna(0).values
alpha_bonf  = 0.05 / K_FINAL

print('Spearman rho: score ~ topic theta loading')
print(f'(* = Bonferroni-significant, threshold p < {alpha_bonf:.4f})')
print()
for k in range(K_FINAL):
    rho, p = spearmanr(score_vals, theta[:, k])
    sig    = '*' if p < alpha_bonf else ''
    print(f'  Topic {k:2d}: rho={rho:+.3f}  p={p:.4f}  {sig}')

## 14. Write LDA Artifacts

In [ ]:
np.save(ARTIFACTS / 'theta.npy', theta)
np.save(ARTIFACTS / 'phi.npy',   phi)

with open(ARTIFACTS / 'lda_vocab.json', 'w') as fh:
    json.dump(list(vocab), fh)

joblib.dump(lda, ARTIFACTS / 'lda_model.pkl')

template = {str(k): f'Topic {k} — label here' for k in range(K_FINAL)}
with open(ARTIFACTS / 'topic_labels.json', 'w') as fh:
    json.dump(template, fh, indent=2)

print('Artifacts written:')
for name in ('posts_clean.parquet', 'theta.npy', 'phi.npy',
             'lda_vocab.json', 'lda_model.pkl', 'topic_labels.json', 'top_docs.md'):
    p = ARTIFACTS / name
    if p.exists():
        print(f'  {p}  ({p.stat().st_size / 1024:.1f} KB)')

print()
print('Fill in topic_labels.json before running downstream notebooks.')

---
# Part 3 — Structural Topic Model (STM) with Time Covariate

STM extends LDA by allowing **topic prevalence** to vary as a function of document-level covariates. Here the covariate is **time** (`day_idx`).

**Implementation:** Pure Python approximation — no R required.
LDA `theta` from Part 2 → daily aggregation → spline smoothing → OLS trend test.
This is mechanistically equivalent to STM's covariate-modulated prevalence at the aggregate level (R `stm` package, `estimateEffect`-style output).

**Dataset:** Dataset 1 (posts-only, `artifacts/posts_only/`)
**Time covariate:** `day_idx` (continuous) and `year_month` (categorical heatmap)

In [ ]:
# ── Attach theta to df if not already present ─────────────────────────────────
# (theta and phi are available from Part 2; reload from disk if kernel restarted)
try:
    _ = theta
except NameError:
    theta = np.load(ARTIFACTS / 'theta.npy')
    phi   = np.load(ARTIFACTS / 'phi.npy')
    with open(ARTIFACTS / 'lda_vocab.json') as _f:
        vocab_list = json.load(_f)
    df = pd.read_parquet(ARTIFACTS / 'posts_clean.parquet')
    # Recompute frex scores (needed for labels)
    from scipy.stats import rankdata as _rankdata
    def frex(phi, w=0.5, n_top=15):
        K, V = phi.shape
        col_sums = phi.sum(axis=0)
        excl = phi / np.where(col_sums > 0, col_sums, 1)
        freq_ecdf = np.apply_along_axis(lambda row: _rankdata(row) / V, axis=1, arr=phi)
        excl_ecdf = np.apply_along_axis(lambda row: _rankdata(row) / V, axis=1, arr=excl)
        scores = 1.0 / (w / freq_ecdf + (1 - w) / excl_ecdf)
        top_idx = scores.argsort(axis=1)[:, ::-1][:, :n_top]
        return scores, top_idx
    frex_scores, frex_top = frex(phi)
    K_FINAL = phi.shape[0]

for k in range(K_FINAL):
    df[f'theta_{k}'] = theta[:, k]

print(f'theta shape: {theta.shape}  |  df shape: {df.shape}')
print(f'Date range : {df["date"].min()} -> {df["date"].max()}')

## 3.1 Topic Quality: Exclusivity vs Coherence

The canonical STM summary plot. Each point = one topic.
- **Coherence** (x-axis): PMI of top words — how often they co-occur
- **Exclusivity** (y-axis): how much of each word's mass is concentrated in this topic
Good topics occupy the upper-right quadrant.

In [ ]:
# Per-topic exclusivity (mean over top-10 words)
col_sums = phi.sum(axis=0)
excl_mat = phi / np.where(col_sums > 0, col_sums, 1)

topic_excl = []
topic_coh  = []
for k in range(K_FINAL):
    top10 = phi[k].argsort()[::-1][:10]
    topic_excl.append(float(excl_mat[k, top10].mean()))
    topic_coh.append(pmi_coherence(k, phi, dtm_csc, n_top=10))

frex_labels = [
    ', '.join(vocab_list[i] for i in frex_top[k][:3])
    for k in range(K_FINAL)
]

fig, ax = plt.subplots(figsize=(7, 5))
colors = plt.cm.tab10.colors

for k in range(K_FINAL):
    ax.scatter(topic_coh[k], topic_excl[k], s=120, color=colors[k], zorder=3)
    ax.annotate(
        f'T{k}: {frex_labels[k]}',
        (topic_coh[k], topic_excl[k]),
        textcoords='offset points', xytext=(6, 4), fontsize=8,
    )

ax.set_xlabel('PMI Coherence (higher = better)')
ax.set_ylabel('Mean Exclusivity (higher = more distinctive)')
ax.set_title('STM Topic Quality: Exclusivity vs Coherence')
ax.axhline(np.mean(topic_excl), color='grey', linestyle='--', lw=0.8, label='mean excl.')
ax.axvline(np.mean(topic_coh),  color='grey', linestyle=':',  lw=0.8, label='mean coh.')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(ARTIFACTS / 'stm_excl_coherence.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Daily mean theta per topic ────────────────────────────────────────────────
theta_cols = [f'theta_{k}' for k in range(K_FINAL)]
daily_theta = (
    df.groupby(['date', 'day_idx'])[theta_cols]
    .mean()
    .reset_index()
    .sort_values('day_idx')
)
print(f'Daily observations: {len(daily_theta)}')
print(daily_theta.head())

## 3.2 Topic Prevalence Over Time (LOWESS Smoothing)

Equivalent to R `stm`'s `plot(fit, type="perspectives", covariate="date")`.

Each panel shows raw daily mean theta (grey dots), a smoothed prevalence curve (`UnivariateSpline`, cubic), and a 95% bootstrap confidence band (500 resamples).

In [ ]:
STM_SMOOTH_FACTOR = 0.06  # controls spline flexibility; lower = more wiggly
STM_N_BOOTSTRAP   = 500
STM_GRID_N        = 300   # points in the smooth curve grid

day_arr   = daily_theta['day_idx'].values.astype(float)
day_grid  = np.linspace(day_arr.min(), day_arr.max(), STM_GRID_N)
date_min  = df['date'].min()
date_grid = pd.to_datetime(date_min) + pd.to_timedelta(day_grid, unit='D')

rng_stm = np.random.default_rng(42)

stm_smooth   = {}  # k -> smooth curve on day_grid
stm_ci_lower = {}
stm_ci_upper = {}

for k in range(K_FINAL):
    y = daily_theta[f'theta_{k}'].values.astype(float)
    s = STM_SMOOTH_FACTOR * len(y)

    # Point estimate
    spline = UnivariateSpline(day_arr, y, k=3, s=s)
    stm_smooth[k] = spline(day_grid)

    # Bootstrap CI
    boot_curves = np.zeros((STM_N_BOOTSTRAP, STM_GRID_N))
    for b in range(STM_N_BOOTSTRAP):
        idx = rng_stm.integers(0, len(day_arr), size=len(day_arr))
        _x, _y = day_arr[idx], y[idx]
        order = np.argsort(_x)
        _x, _y = _x[order], _y[order]
        try:
            _sp = UnivariateSpline(_x, _y, k=3, s=s)
            boot_curves[b] = _sp(day_grid)
        except Exception:
            boot_curves[b] = np.full(STM_GRID_N, y.mean())

    stm_ci_lower[k] = np.percentile(boot_curves, 2.5,  axis=0)
    stm_ci_upper[k] = np.percentile(boot_curves, 97.5, axis=0)

print('Spline fitting complete.')

In [ ]:
# ── Main STM prevalence figure ────────────────────────────────────────────────
ncols = 2
nrows = (K_FINAL + 1) // 2
fig, axes = plt.subplots(nrows, ncols, figsize=(13, 4 * nrows), sharex=False)
axes = axes.flatten() if K_FINAL > 1 else [axes]

colors = plt.cm.tab10.colors

for k in range(K_FINAL):
    ax = axes[k]
    y_raw = daily_theta[f'theta_{k}'].values
    x_raw = daily_theta['day_idx'].values
    date_raw = pd.to_datetime(date_min) + pd.to_timedelta(x_raw, unit='D')

    # Raw daily points
    ax.scatter(date_raw, y_raw, color='lightgrey', s=12, zorder=1, label='Daily mean')

    # Smooth curve
    ax.plot(date_grid, stm_smooth[k], color=colors[k], lw=2, zorder=3, label='Spline')

    # Bootstrap CI band
    ax.fill_between(
        date_grid,
        stm_ci_lower[k], stm_ci_upper[k],
        color=colors[k], alpha=0.15, zorder=2, label='95% CI',
    )

    # FREX top-5 as title
    top5 = ', '.join(vocab_list[i] for i in frex_top[k][:5])
    ax.set_title(f'Topic {k}: {top5}', fontsize=9, pad=4)
    ax.set_ylabel('Mean theta')
    ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%b %d'))
    ax.xaxis.set_major_locator(plt.matplotlib.dates.WeekdayLocator(interval=2))
    plt.setp(ax.get_xticklabels(), rotation=35, ha='right', fontsize=7)
    ax.legend(fontsize=7, loc='upper right')

# Hide unused panels
for j in range(K_FINAL, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('STM: Topic Prevalence Over Time (posts-only corpus)', fontsize=11, y=1.01)
plt.tight_layout()
plt.savefig(ARTIFACTS / 'stm_prevalence_by_topic.png', dpi=150, bbox_inches='tight')
plt.show()

## 3.3 OLS Linear Trend Test

For each topic, fit OLS on daily mean theta vs `day_idx`.
Slope and R² quantify the linear temporal trend; p-values are Bonferroni-corrected.

In [ ]:
ols_results = []
alpha_bonf_stm = 0.05 / K_FINAL

print('OLS linear trend: mean_theta ~ day_idx')
print(f'(* = Bonferroni-significant, threshold p < {alpha_bonf_stm:.4f})')
print()

for k in range(K_FINAL):
    y = daily_theta[f'theta_{k}'].values.astype(float)
    x = daily_theta['day_idx'].values.astype(float)
    slope, intercept, r, p, se = linregress(x, y)
    sig = '*' if p < alpha_bonf_stm else ''
    top3 = ', '.join(vocab_list[i] for i in frex_top[k][:3])
    print(f'  Topic {k} ({top3}):  slope={slope:+.6f}  R²={r**2:.4f}  p={p:.4f}  {sig}')
    ols_results.append({
        'topic': k, 'frex_top3': top3,
        'slope': slope, 'r2': r**2, 'p': p,
        'significant': p < alpha_bonf_stm,
    })

ols_df = pd.DataFrame(ols_results)
ols_df.to_csv(ARTIFACTS / 'stm_ols_results.csv', index=False)
print(f'\nSaved -> {ARTIFACTS / "stm_ols_results.csv"}')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 0.8 * K_FINAL + 2))
colors_sig = ['#e74c3c' if row['significant'] else '#95a5a6' for _, row in ols_df.iterrows()]
y_pos = range(K_FINAL)
labels = [f'T{r.topic}: {r.frex_top3}' for _, r in ols_df.iterrows()]

ax.barh(y_pos, ols_df['slope'], color=colors_sig, height=0.5)
ax.axvline(0, color='black', lw=0.8)
ax.set_yticks(list(y_pos))
ax.set_yticklabels(labels, fontsize=9)
ax.set_xlabel('OLS slope (change in mean theta per day)')
ax.set_title('STM Linear Time Trend per Topic\n(red = Bonferroni-significant)')
plt.tight_layout()
plt.savefig(ARTIFACTS / 'stm_ols_slopes.png', dpi=150, bbox_inches='tight')
plt.show()

## 3.4 Monthly Prevalence Heatmap

Aggregate theta by `year_month`. Z-scored per topic column so each topic's variation is on the same scale regardless of base rate.

In [ ]:
monthly_theta = (
    df.groupby('year_month')[theta_cols].mean()
    .sort_index()
)

# Z-score each topic column independently
monthly_z = (monthly_theta - monthly_theta.mean()) / (monthly_theta.std() + 1e-12)

topic_col_labels = [
    f'T{k}: {", ".join(vocab_list[i] for i in frex_top[k][:3])}'
    for k in range(K_FINAL)
]

fig, ax = plt.subplots(figsize=(max(6, K_FINAL * 1.8), max(3, len(monthly_theta) * 0.8)))
sns.heatmap(
    monthly_z,
    annot=True, fmt='.2f', cmap='RdBu_r', center=0,
    xticklabels=topic_col_labels,
    yticklabels=monthly_theta.index.tolist(),
    ax=ax,
)
ax.set_title('STM: Monthly Topic Prevalence (z-scored per topic)')
ax.set_xlabel('')
ax.set_ylabel('Month')
plt.setp(ax.get_xticklabels(), rotation=25, ha='right', fontsize=8)
plt.tight_layout()
plt.savefig(ARTIFACTS / 'stm_monthly_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 3.5 Flair Covariate Effect

Categorical analogue of STM's `estimateEffect`. Mean topic loading per flair category — shows whether certain content types are systematically associated with particular topics.

In [ ]:
flair_counts = df['flair'].value_counts()
TOP_FLAIRS   = flair_counts[flair_counts >= 5].index.tolist()[:12]  # cap at 12

if len(TOP_FLAIRS) <= 1:
    print('Not enough flair variety to plot (all posts share one flair or flair is missing).')
else:
    flair_theta = (
        df[df['flair'].isin(TOP_FLAIRS)]
        .groupby('flair')[theta_cols]
        .mean()
        .loc[TOP_FLAIRS]
    )

    x    = np.arange(len(TOP_FLAIRS))
    w    = 0.8 / K_FINAL
    fig, ax = plt.subplots(figsize=(max(9, len(TOP_FLAIRS) * 0.9), 4))
    colors = plt.cm.tab10.colors

    for k in range(K_FINAL):
        offset = (k - K_FINAL / 2 + 0.5) * w
        top3 = ', '.join(vocab_list[i] for i in frex_top[k][:3])
        ax.bar(x + offset, flair_theta[f'theta_{k}'], width=w * 0.9,
               color=colors[k], label=f'T{k}: {top3}', alpha=0.85)

    ax.set_xticks(x)
    ax.set_xticklabels(TOP_FLAIRS, rotation=35, ha='right', fontsize=8)
    ax.set_ylabel('Mean theta loading')
    ax.set_title('STM Flair Effect: Mean Topic Prevalence by Post Flair')
    ax.legend(fontsize=7, ncol=2)
    plt.tight_layout()
    plt.savefig(ARTIFACTS / 'stm_flair_effect.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# ── Save STM prevalence table ──────────────────────────────────────────────────
stm_export = daily_theta.copy()
for k in range(K_FINAL):
    stm_export[f'smooth_{k}'] = stm_smooth[k][
        np.searchsorted(day_grid, daily_theta['day_idx'].values.clip(day_grid.min(), day_grid.max()))
        .clip(0, STM_GRID_N - 1)
    ]
stm_export.to_csv(ARTIFACTS / 'stm_daily_prevalence.csv', index=False)
print(f'Saved -> {ARTIFACTS / "stm_daily_prevalence.csv"}')
print('STM Part 3 complete.')

---
# Part 4 — sLDA-C (Supervised LDA, Continuous Response)

Two-stage sLDA: fit LDA on the richer combined post+comment documents (Dataset 2), then regress engagement metrics on the resulting document-topic distributions.

**Responses:** `log_score = log1p(score)`, `log_num_comments = log1p(num_comments)`
**Model:** `RidgeCV` with 5-fold CV — alpha selected automatically.

> **Limitation:** Two-stage estimation does not propagate theta uncertainty into regression standard errors. Interpret coefficients as *associations*, not causal effects.

**Dataset:** `artifacts/posts_comments/posts_with_comments.parquet`

In [ ]:
# ── Load Dataset 2 ─────────────────────────────────────────────────────────────
try:
    _ = df2
    print(f'df2 already in memory: {len(df2):,} rows')
except NameError:
    df2 = pd.read_parquet(ARTIFACTS_D2 / 'posts_with_comments.parquet')
    print(f'Loaded df2 from parquet: {len(df2):,} rows')

print(f'log_score        — mean={df2["log_score"].mean():.3f}, std={df2["log_score"].std():.3f}')
print(f'log_num_comments — mean={df2["log_num_comments"].mean():.3f}, std={df2["log_num_comments"].std():.3f}')

## 4.1 Document-Term Matrix (Combined Documents)

Same pipeline as Part 2 but on `combined_text_lemma`. Raises `min_df` to 15 because the larger combined vocabulary would otherwise include very rare comment terms.

In [ ]:
vec2 = CountVectorizer(min_df=15, max_df=0.90)
dtm2 = vec2.fit_transform(df2['combined_text_lemma'])
vocab2     = vec2.get_feature_names_out()
vocab2_list = list(vocab2)

dtm2_csc = csc_matrix((dtm2 > 0).astype(np.float32))

X2_train, X2_test = train_test_split(dtm2, test_size=0.10, random_state=42)

print(f'DTM2     : {dtm2.shape[0]:,} docs × {dtm2.shape[1]:,} terms')
print(f'Sparsity : {1 - dtm2.nnz / (dtm2.shape[0] * dtm2.shape[1]):.2%}')
print(f'Train: {X2_train.shape[0]:,}   Test: {X2_test.shape[0]:,}')

## 4.2 K Selection for sLDA-C

Three criteria:
- **Perplexity** on held-out 10% (LDA reconstruction quality)
- **PMI coherence** (topic interpretability)
- **CV R²** of `Ridge(log_score ~ theta)` — predictive signal from topics to engagement

In [ ]:
K_RANGE2    = [2, 3, 4, 5, 6, 7, 8]
K2_results  = []
_kf_k       = KFold(n_splits=5, shuffle=True, random_state=42)

log_score_arr = df2['log_score'].values

for K in K_RANGE2:
    _lda_k = LatentDirichletAllocation(
        n_components=K, learning_method='online', max_iter=20, random_state=42
    )
    _lda_k.fit(X2_train)
    _perp = _lda_k.perplexity(X2_test)
    _phi_k = _lda_k.components_ / _lda_k.components_.sum(axis=1, keepdims=True)
    _coh   = float(np.mean([pmi_coherence(k, _phi_k, dtm2_csc) for k in range(K)]))

    # CV R² for Ridge regression
    _theta_k = _lda_k.transform(dtm2)
    _ridge_k = RidgeCV(alphas=np.logspace(-3, 3, 10), cv=3)
    _y_pred  = cross_val_predict(_ridge_k, _theta_k, log_score_arr, cv=_kf_k)
    _r2_cv   = r2_score(log_score_arr, _y_pred)

    K2_results.append({'K': K, 'perplexity': _perp, 'coherence': _coh, 'cv_r2': _r2_cv})
    print(f'K={K}  perp={_perp:9.1f}  coh={_coh:.4f}  CV-R²={_r2_cv:.4f}')

K2_df = pd.DataFrame(K2_results)

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(13, 4))

ax1.plot(K2_df['K'], K2_df['perplexity'], marker='o', color='steelblue')
ax1.set_title('Perplexity (held-out 10%)')
ax1.set_xlabel('K'); ax1.set_ylabel('Perplexity (lower = better)')

ax2.plot(K2_df['K'], K2_df['coherence'], marker='o', color='darkorange')
ax2.set_title('PMI Coherence')
ax2.set_xlabel('K'); ax2.set_ylabel('Mean PMI (higher = better)')

ax3.plot(K2_df['K'], K2_df['cv_r2'], marker='o', color='mediumseagreen')
ax3.axhline(0, color='grey', lw=0.8, linestyle='--')
ax3.set_title('5-fold CV R² (Ridge: log_score ~ theta)')
ax3.set_xlabel('K'); ax3.set_ylabel('CV R²')

plt.suptitle('sLDA-C K Selection — choose K balancing coherence and predictive R²')
plt.tight_layout()
plt.show()

## 4.3 Fit Final sLDA LDA

Set `K_SLDA` based on the plot above (defaults to `K_FINAL` from Part 2).

In [ ]:
K_SLDA = K_FINAL  # ← adjust based on K selection plot above

lda2 = LatentDirichletAllocation(
    n_components=K_SLDA, learning_method='batch', max_iter=100, random_state=42
)
lda2.fit(dtm2)

phi2   = lda2.components_ / lda2.components_.sum(axis=1, keepdims=True)
theta2 = lda2.transform(dtm2)

print(f'K_SLDA={K_SLDA}  |  phi2: {phi2.shape}  |  theta2: {theta2.shape}')
print(f'Held-out perplexity: {lda2.perplexity(X2_test):.1f}')

## 4.4 FREX Profiling (Dataset 2 Topics)

Compare with Dataset 1 (posts-only) FREX words. Topics are matched by maximum word overlap — shows which themes persist with vs without comment text.

In [ ]:
frex_scores2, frex_top2 = frex(phi2, w=FREX_W, n_top=N_FREX)

print(f'FREX top-{N_FREX} words per sLDA topic  (Dataset 2)')
print('=' * 70)
for k in range(K_SLDA):
    words = ', '.join(vocab2_list[i] for i in frex_top2[k])
    print(f'Topic {k}: {words}')

# ── Align sLDA topics with LDA (Dataset 1) by FREX word overlap ───────────────
print('\n\nTopic alignment (Dataset 1 LDA -> Dataset 2 sLDA by FREX top-15 overlap):')
print('-' * 60)
d1_sets = {k: set(vocab_list[i] for i in frex_top[k]) for k in range(K_FINAL)}

for k2 in range(K_SLDA):
    s2 = set(vocab2_list[i] for i in frex_top2[k2])
    overlaps = {k1: len(s2 & d1_sets[k1]) for k1 in range(K_FINAL)}
    best_k1  = max(overlaps, key=overlaps.get)
    shared   = sorted(s2 & d1_sets[best_k1])
    print(f'  sLDA T{k2} -> LDA T{best_k1}  ({overlaps[best_k1]} shared FREX words: {", ".join(shared[:6])}...)')

## 4.5 Ridge Regression: Topic Loadings → Engagement

Two regressions:
1. `log_score ~ theta2` — does topic predict post score?
2. `log_num_comments ~ theta2` — does topic predict comment volume?

Both use 5-fold CV `RidgeCV` with auto alpha selection.

In [ ]:
kf2 = KFold(n_splits=5, shuffle=True, random_state=42)

# ── Score regression ───────────────────────────────────────────────────────────
ridge_score2 = RidgeCV(alphas=np.logspace(-3, 3, 20), cv=kf2)
ridge_score2.fit(theta2, df2['log_score'])
y_pred_score2    = cross_val_predict(ridge_score2, theta2, df2['log_score'], cv=kf2)
cv_r2_score2     = r2_score(df2['log_score'], y_pred_score2)
insample_r2_score = ridge_score2.score(theta2, df2['log_score'])

print('=== log_score ~ theta2 ===')
print(f'  Alpha selected : {ridge_score2.alpha_:.4f}')
print(f'  In-sample R²   : {insample_r2_score:.4f}')
print(f'  5-fold CV R²   : {cv_r2_score2:.4f}')
print()
for k in range(K_SLDA):
    top3 = ', '.join(vocab2_list[i] for i in frex_top2[k][:3])
    print(f'  T{k} ({top3}): coef = {ridge_score2.coef_[k]:+.4f}')

print()

# ── num_comments regression ────────────────────────────────────────────────────
ridge_nc2 = RidgeCV(alphas=np.logspace(-3, 3, 20), cv=kf2)
ridge_nc2.fit(theta2, df2['log_num_comments'])
y_pred_nc2    = cross_val_predict(ridge_nc2, theta2, df2['log_num_comments'], cv=kf2)
cv_r2_nc2     = r2_score(df2['log_num_comments'], y_pred_nc2)
insample_r2_nc = ridge_nc2.score(theta2, df2['log_num_comments'])

print('=== log_num_comments ~ theta2 ===')
print(f'  Alpha selected : {ridge_nc2.alpha_:.4f}')
print(f'  In-sample R²   : {insample_r2_nc:.4f}')
print(f'  5-fold CV R²   : {cv_r2_nc2:.4f}')
print()
for k in range(K_SLDA):
    top3 = ', '.join(vocab2_list[i] for i in frex_top2[k][:3])
    print(f'  T{k} ({top3}): coef = {ridge_nc2.coef_[k]:+.4f}')

## 4.6 Topic-Response Coefficient Plot

In [ ]:
frex_labels2 = [
    f'T{k}: {", ".join(vocab2_list[i] for i in frex_top2[k][:3])}'
    for k in range(K_SLDA)
]
y_pos2 = np.arange(K_SLDA)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, max(3, K_SLDA * 0.7 + 1.5)), sharey=True)

def _bar_colors(coefs):
    return ['#e74c3c' if c > 0 else '#3498db' for c in coefs]

ax1.barh(y_pos2, ridge_score2.coef_, color=_bar_colors(ridge_score2.coef_), height=0.5)
ax1.axvline(0, color='black', lw=0.8)
ax1.set_yticks(y_pos2); ax1.set_yticklabels(frex_labels2, fontsize=8)
ax1.set_xlabel('Ridge coefficient')
ax1.set_title(f'log_score  (CV R²={cv_r2_score2:.3f})')

ax2.barh(y_pos2, ridge_nc2.coef_, color=_bar_colors(ridge_nc2.coef_), height=0.5)
ax2.axvline(0, color='black', lw=0.8)
ax2.set_xlabel('Ridge coefficient')
ax2.set_title(f'log_num_comments  (CV R²={cv_r2_nc2:.3f})')

plt.suptitle('sLDA-C: Topic Coefficients\n(red=positive, blue=negative)', fontsize=11)
plt.tight_layout()
plt.savefig(ARTIFACTS_D2 / 'slda_topic_coefficients.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
ncols2 = 2
nrows2 = (K_SLDA + 1) // 2
fig2, axes2 = plt.subplots(nrows2, ncols2, figsize=(11, 4 * nrows2))
axes2 = axes2.flatten() if K_SLDA > 1 else [axes2]

for k in range(K_SLDA):
    ax = axes2[k]
    top3 = ', '.join(vocab2_list[i] for i in frex_top2[k][:3])
    import seaborn as _sns
    _sns.regplot(
        x=theta2[:, k], y=df2['log_score'].values,
        ax=ax, scatter_kws={'alpha': 0.08, 's': 10, 'color': 'steelblue'},
        line_kws={'color': 'crimson', 'lw': 1.5},
    )
    ax.set_xlabel(f'theta2 (T{k})')
    ax.set_ylabel('log_score')
    ax.set_title(f'T{k}: {top3}', fontsize=8)

for j in range(K_SLDA, len(axes2)):
    axes2[j].set_visible(False)

plt.suptitle('sLDA-C: Topic Loading vs log_score', fontsize=11)
plt.tight_layout()
plt.savefig(ARTIFACTS_D2 / 'slda_topic_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

## 4.7 Top Documents by Predicted Engagement

In [ ]:
y_hat_score = ridge_score2.predict(theta2)
df2_inspect = df2.copy()
df2_inspect['pred_log_score']  = y_hat_score
df2_inspect['dom_topic']       = theta2.argmax(axis=1)

# Topic with largest positive coefficient
best_k  = int(ridge_score2.coef_.argmax())
worst_k = int(ridge_score2.coef_.argmin())

for label, k in [('Highest-engagement topic', best_k), ('Lowest-engagement topic', worst_k)]:
    top3 = ', '.join(vocab2_list[i] for i in frex_top2[k][:3])
    print(f'\n{"="*70}')
    print(f'{label}: T{k} ({top3}) | coef={ridge_score2.coef_[k]:+.4f}')
    print('='*70)
    sub = df2_inspect[df2_inspect['dom_topic'] == k].nlargest(5, 'pred_log_score')
    for _, row in sub.iterrows():
        print(f'  score={row["score"]}  pred={row["pred_log_score"]:.2f}')
        snippet = str(row.get('combined_text_lemma', ''))[:200]
        print(f'  {snippet}...')
        print()

## 4.8 Residual Analysis

In [ ]:
residuals2 = df2['log_score'].values - y_pred_score2

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.scatter(y_pred_score2, residuals2, alpha=0.08, s=8, color='steelblue')
ax1.axhline(0, color='red', lw=1)
ax1.set_xlabel('Fitted log_score')
ax1.set_ylabel('Residuals')
ax1.set_title('Residuals vs Fitted (log_score)')

ax2.hist(residuals2, bins=50, color='steelblue', edgecolor='none')
ax2.axvline(0, color='red', lw=1)
ax2.set_xlabel('Residual')
ax2.set_title('Residual Distribution')

plt.suptitle('sLDA-C Residual Diagnostics')
plt.tight_layout()
plt.show()

if residuals2.std() > 0:
    from scipy.stats import spearmanr as _spr
    rho_res, p_res = _spr(np.abs(residuals2), y_pred_score2)
    print(f'Heteroscedasticity check — Spearman(|residual|, fitted): rho={rho_res:.3f}, p={p_res:.4f}')
    if p_res < 0.05:
        print('Significant fan-out pattern detected. Consider Poisson/NegBin GLM for count outcomes.')

In [ ]:
np.save(ARTIFACTS_D2 / 'slda_theta.npy', theta2)
np.save(ARTIFACTS_D2 / 'slda_phi.npy',   phi2)

with open(ARTIFACTS_D2 / 'slda_vocab.json', 'w') as _f:
    json.dump(vocab2_list, _f)

joblib.dump(lda2,         ARTIFACTS_D2 / 'slda_lda_model.pkl')
joblib.dump(ridge_score2, ARTIFACTS_D2 / 'slda_ridge_score.pkl')
joblib.dump(ridge_nc2,    ARTIFACTS_D2 / 'slda_ridge_comments.pkl')

# Summary CSV
slda_results_rows = []
for k in range(K_SLDA):
    slda_results_rows.append({
        'topic': k,
        'frex_top5': ', '.join(vocab2_list[i] for i in frex_top2[k][:5]),
        'coef_log_score': ridge_score2.coef_[k],
        'coef_log_num_comments': ridge_nc2.coef_[k],
        'cv_r2_score': cv_r2_score2,
        'cv_r2_num_comments': cv_r2_nc2,
    })
pd.DataFrame(slda_results_rows).to_csv(ARTIFACTS_D2 / 'slda_results.csv', index=False)

print('sLDA-C artifacts written:')
for name in ('slda_theta.npy', 'slda_phi.npy', 'slda_vocab.json',
             'slda_lda_model.pkl', 'slda_ridge_score.pkl',
             'slda_ridge_comments.pkl', 'slda_results.csv',
             'slda_topic_coefficients.png', 'slda_topic_scatter.png'):
    p = ARTIFACTS_D2 / name
    if p.exists():
        print(f'  {p}  ({p.stat().st_size / 1024:.1f} KB)')
print('\nPart 4 sLDA-C complete.')